In [5]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('usa_houses.csv')
df = df[["price",'sqft_living', 'bedrooms', 'floors', 'view']]
df = df[(df['price'] < 2700000) & (df['price'] > 10000)]

features_df = df[['sqft_living', 'bedrooms', 'floors', 'view']]
target_df = df[["price"]]

standardized_polynomial_features_df = MinMaxScaler().fit_transform(features_df)

## 🎯 Model Optimization: Parameters vs. Hyperparameters

Before we can optimize a model effectively, we must clearly distinguish between two key concepts: **parameters** and **hyperparameters**.

### 📌 Model Parameters
- **Definition**: Internal values that the model automatically learns from the data during the training process.
- **Role**: They represent the actual *knowledge* acquired by the model.
- **Learning Process**: Parameters are updated during optimization to minimize a cost function (e.g., Mean Squared Error).

### ⚙️ Model Hyperparameters
- **Definition**: External configurations that are **not** learned from the data.
- **Role**: They act as *rules* that guide the learning process.
- **Setup**: Must be set **before** training begins.
- **Impact**: The choice of hyperparameters directly influences the model’s final performance.
- **Examples**:
  - The **degree** of a polynomial in polynomial regression.
  - The **alpha** value in Ridge or Lasso regression.


- **Parameters** → learned automatically from data.  
- **Hyperparameters** → chosen manually to control how the model learns.

## 🔧 Machine Learning Optimization: Hyperparameter Tuning

Hyperparameter tuning is the process of searching for and selecting the optimal values of a model’s hyperparameters in order to maximize its performance.

### 📌 Key Points
- There is **no universal formula** for choosing hyperparameters — the best combination depends on the dataset and the specific problem.
- Hyperparameters must be set **before** training; they are not learned from the data.
- The tuning process can be **computationally expensive**, especially for large search spaces.

### 🛠 Typical Workflow
1. **Select** a set of hyperparameters to test.
2. **Train** the model with each combination of values.
3. **Evaluate** performance on a separate dataset:
   - Use a **validation set** or
   - Apply **cross‑validation**  
4. **Repeat** for all combinations in the search space.
5. **Select** the model that achieves the best score on the validation metric.

### 💡 Example Hyperparameters
- Degree of a polynomial in polynomial regression.
- `alpha` in Ridge or Lasso regression.
- `l1_ratio` in ElasticNet.
- Number of trees in a Random Forest.
- Learning rate in gradient boosting.

**Goal**: Find the sweet spot where the model generalizes well — not underfitting, not overfitting — by systematically exploring the hyperparameter space.

## ⚠️ The Validation Set Problem

We’ve learned to split our dataset into three parts: **training set**, **validation set**, and **test set**.  
This strategy works well in many cases, but it has **limitations**, especially when working with **small datasets**.
Splitting into three sets can leave **too few samples** for effective training, the **validation set** might not be  
**representative** of the full data distribution.
Model performance may depend **heavily** on which random samples end up in the validation set.

### 🧠 Why This Matters
- A poorly chosen validation set can lead to **biased performance estimates**.
- The model might appear better or worse than it truly is, simply due to the randomness of the split.
- This undermines the reliability of our evaluation and can mislead hyperparameter tuning.

### ✅ How to Improve Robustness
To obtain a **more reliable and stable evaluation**, we can use:

- **Cross‑Validation (CV)**:  
  Instead of relying on a single validation set, we rotate through multiple training/validation splits, we try to kill randomness.
- This ensures that **every data point** is used for both training and validation — just not at the same time.

While the train/validation/test split is useful, it can be fragile with small datasets.  
Cross‑validation offers a more **robust and fair** way to assess model performance, so let's explore it.

## 📊 Cross‑Validation (CV) — A Robust Evaluation Method

Cross‑validation is a statistical technique used to estimate a model’s performance in a more robust way.  
The key idea: use **all available data** for both training and validation, but in **separate phases**.

### 🔹 Why Use Cross‑Validation?
- Reduces the risk of **overestimating** performance compared to a single train/test split.
- Ensures the evaluation is **less dependent** on how the data is split.
- Makes better use of limited datasets.

### 🔹 The Most Common Approach: *k‑Fold Cross‑Validation*
1. **Split** the dataset into `k` equally sized blocks (called *folds*).
2. For each iteration:
   - Select **1 fold** as the **validation set**.
   - Use the remaining **k−1 folds** as the **training set**.
3. **Repeat** this process `k` times, so that each fold is used exactly once as the validation set.
4. **Aggregate** results: the final performance metric (e.g., MSE, RMSE, R²) is the **average** of the metrics from all `k` iterations.

### 💡 Example
- **k = 5**: The dataset is split into 5 folds.
- The model is trained and validated 5 times, each time with a different fold as the validation set.
- The reported score is the **mean** of the 5 validation scores.

Cross‑validation provides a **more reliable estimate** of how the model will perform on unseen data, especially when the dataset is small or when variance in performance is high.

In [6]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Step 1: Create polynomial features manually
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(standardized_polynomial_features_df)

# Step 2: Define the model
model = LinearRegression()

# Step 3: Set up K-Fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Step 4: Run cross-validation
r2_scores = cross_val_score(model, X_poly, target_df, cv=cv, scoring="r2")
mse_scores = cross_val_score(model, X_poly, target_df, cv=cv, scoring="neg_mean_squared_error")

# Step 5: Convert to RMSE
print("Average R²:", r2_scores.mean())
rmse_scores = np.sqrt(-mse_scores)

print("Average RMSE:", np.mean(rmse_scores))


Average R²: 0.49878043833666774
Average RMSE: 231300.8199827168


## 🔍 Leave-One-Out Cross-Validation (LOOCV)

### 📘 Definition  
Leave-One-Out Cross-Validation (LOOCV) is a special case of k-fold cross-validation where the number of folds `k` equals the number of data points `n`.  
In each round, the model is trained on `n−1` samples and tested on the **one** sample left out.  
This process is repeated `n` times — once for every data point.

### ✅ Why It’s Useful
- **Uses almost all the data** for training in each round.
- **No randomness**: Every data point gets tested exactly once, so results are consistent across runs.

### ⚠️ What to Watch Out For
- **Very slow**: The model has to be trained `n` times — which can be a lot if your dataset is big.
- **Can be noisy**: Because each training set is nearly identical, the performance estimates might jump around more than with regular k-fold CV.

### 🧠 When to Use It
- Great for **small datasets** where every observation matters.
- Useful when you want a **stable and unbiased estimate** of model performance — as long as your machine can handle the extra work.

LOOCV is like interrogating each data of your model — everyone gets an occasion to show the model their truth, and the model can learn from all of them equally.

In [7]:
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.linear_model import LinearRegression
import numpy as np

model = LinearRegression()
cv = LeaveOneOut()

# Use scoring="neg_mean_squared_error" instead of "r2"
mse_scores = cross_val_score(model, standardized_polynomial_features_df, target_df, cv=cv, scoring="neg_mean_squared_error")

# Convert to RMSE
rmse_scores = np.sqrt(-mse_scores)

print("Average RMSE:", np.mean(rmse_scores))

Average RMSE: 159822.47237472745
